# 08 · Function Approximation
**DEEP RL** · ⏱ 45 min · 🧪 3 exercises · ⭐ 440 XP

*Replace the table with a parameterised function — features, linear values, semi-gradient TD, tile coding, and the deadly triad.*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "08-function-approximation"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

Every method so far kept **one number per state** (or per state–action pair) in a table. That
works for a 4×12 cliff. It does not work for:

- **CartPole** — four *continuous* numbers. There is no "state 17"; you will never see the
  exact same state twice.
- **Backgammon** — about 10²⁰ states. **Go** — about 10¹⁷⁰. A table would not fit in the universe.
- **Atari from pixels** — 256^(210·160·3) possible screens.

The real problem is not memory, it is **generalisation**. A table learns nothing about state
*B* from a visit to state *A*, even if they are nearly identical. To learn in large spaces the
agent must assume that *similar states have similar values* — and that is exactly what a
function approximator does.

```text
v̂(s, w) ≈ v_π(s)          q̂(s, a, w) ≈ q_π(s, a)          w ∈ ℝᵈ,  d ≪ |S|
```

Changing one weight now changes the estimate of *many* states at once. That is the power
(generalisation) and the danger (an update for one state can wreck another).

## 01 · Prediction as (almost) supervised learning

If an oracle gave us the true `v_π(s)`, we would just do regression. We want weights that
minimise the **mean squared value error**, weighted by how often each state is visited (μ):

```text
VE(w) = Σₛ μ(s) [ v_π(s) − v̂(s, w) ]²
```

Stochastic gradient descent on one sample *Sₜ* gives

```text
w ← w + α [ v_π(Sₜ) − v̂(Sₜ, w) ] ∇v̂(Sₜ, w)
```

We do not know `v_π(Sₜ)`, so we plug in a **target** *Uₜ* that estimates it — the same targets
you already know from the tabular chapters:

| method | target Uₜ | unbiased? |
|---|---|---|
| gradient Monte Carlo | Gₜ (the full return) | yes — true SGD, converges to a local minimum of VE |
| semi-gradient TD(0) | Rₜ₊₁ + γ v̂(Sₜ₊₁, w) | no — the target uses the current w |
| semi-gradient n-step TD | Rₜ₊₁ + … + γⁿ⁻¹Rₜ₊ₙ + γⁿ v̂(Sₜ₊ₙ, w) | no |

> **ℹ️ NOTE** · **Why "semi"-gradient?**
>
> The TD target contains w too. A *full* gradient would differentiate through it. We
> deliberately don't: we treat the target as a fixed label and only follow the gradient of the
> prediction. It is only "half" of the true gradient — hence *semi*-gradient. It learns faster
> and works well in practice, but it is not SGD on any objective, and that will come back to bite us.

## 02 · Linear value functions

The simplest, best-understood approximator is **linear** in a feature vector x(s) ∈ ℝᵈ:

```text
v̂(s, w) = wᵀ x(s) = Σᵢ wᵢ xᵢ(s)          ∇v̂(s, w) = x(s)
```

The gradient is just the features, so every update has the same shape:

```text
w ← w + α · (target − v̂(Sₜ, w)) · x(Sₜ)
```

A tabular method is the special case where x(s) is a one-hot vector with one entry per state.
*All* the intelligence lives in the choice of features.

### Worked example

Two features, w = [0, 1], α = 0.1, γ = 0.9. We are in S with x(S) = [1, 0], receive r = 0.5,
and land in S′ with x(S′) = [0, 1].

- prediction v̂(S) = 0·1 + 1·0 = 0
- target r + γ v̂(S′) = 0.5 + 0.9 · 1 = 1.4
- TD error δ = 1.4 − 0 = 1.4
- update w ← [0, 1] + 0.1 · 1.4 · [1, 0] = **[0.14, 1]**

Only the weight of the feature that was *active* in S moved. With overlapping features, several
weights move — and every state that shares them changes value too.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("08-function-approximation#0", {"q": "With a linear v̂(s, w) = wᵀx(s), what is ∇v̂(s, w)?", "options": ["w", "x(s)", "wᵀx(s) · x(s)", "It depends on the TD error"], "answer": 1, "explain": "The derivative of Σᵢ wᵢxᵢ(s) with respect to wᵢ is xᵢ(s). That is why linear methods are so cheap: the gradient is the feature vector itself."})

### Gradient MC vs semi-gradient TD

On the 1000-state random walk (jump up to 100 states left or right; −1 off the left edge, +1
off the right) with **state aggregation** — 10 groups of 100 states, one weight per group —
both methods learn a staircase approximation of a nearly straight line.

- **Gradient MC** converges to the *best* staircase (minimum VE), but slowly: its targets are
  noisy ±1 outcomes, so α must be small.
- **Semi-gradient TD(0)** converges to the **TD fixed point**, which is close to — but not
  exactly — the minimum. For linear TD there is a guarantee: VE(w_TD) ≤ 1/(1−γ) · min VE.
  In exchange it learns from every step with far less variance.

---
### 🧪 Exercise 1 · Linear values: gradient MC & semi-gradient TD(0) · ⭐ **120 XP**

Implement the two workhorse prediction updates for a **linear** value function v̂(s, w) = wᵀx(s):

- `v_hat(w, x)` — the dot product.
- `gradient_mc_update(w, xs, rewards, alpha, gamma)` — for one finished episode, compute every return Gₜ and apply **w ← w + α [Gₜ − v̂(Sₜ, w)] x(Sₜ)** for each visited state.
- `td0_update(w, x, r, x_next, terminated, alpha, gamma)` — one **semi-gradient TD(0)** step: target = r (if terminated) or r + γ v̂(S′, w); δ = target − v̂(S, w); **w ← w + α δ x(S)**. Return δ.

Update `w` **in place** (numpy `+=`). The provided code runs both on the 1000-state random walk (Sutton & Barto Example 9.1) with *state aggregation*: 10 groups of 100 states, one weight per group. The check compares your learned values with the exact ones.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 08-function-approximation/01-linear-value-updates
import numpy as np
import rlhud


# ---------------------------------------------------------------- your code
def v_hat(w, x):
    """Linear value estimate: v̂(s, w) = wᵀ x(s)."""
    # TODO: one line
    raise NotImplementedError


def gradient_mc_update(w, xs, rewards, alpha, gamma=1.0):
    """Gradient Monte Carlo for ONE finished episode. Updates w in place and returns it.

    xs      : feature vectors [x(S0), x(S1), ..., x(S_{T-1})]
    rewards : [R1, R2, ..., RT]   (rewards[t] follows the action taken in S_t)
    For every t:  w ← w + α [Gₜ − v̂(Sₜ, w)] ∇v̂(Sₜ, w),  and for a linear v̂, ∇v̂ = x(Sₜ).
    """
    # TODO: 1) compute the return Gₜ for every t (walk backwards: G = r + γ G)
    #       2) for each (x, Gₜ): nudge w toward Gₜ along x
    raise NotImplementedError


def td0_update(w, x, r, x_next, terminated, alpha, gamma=1.0):
    """Semi-gradient TD(0) for one transition S --r--> S'. Updates w in place, returns δ.

    target = r                      if the episode terminated at S'
             r + γ v̂(S', w)         otherwise          (no gradient flows through the target)
    """
    # TODO: compute the target, the TD error δ = target − v̂(S, w), then w += α δ x
    raise NotImplementedError


# ---------------------------------------------------------------- provided: the 1000-state random walk
class RandomWalk1000:
    """States 0..999, start in the middle. Each step jumps 1..100 states left or right
    (uniformly). Falling off the left edge ends the episode with reward −1, the right edge +1.
    All other rewards are 0. Sutton & Barto, Example 9.1."""

    n_states = 1000

    def __init__(self, seed=None):
        self.rng = np.random.default_rng(seed)

    def reset(self):
        self.s = 499
        return self.s

    def step(self):
        jump = int(self.rng.integers(1, 101)) * (1 if self.rng.random() < 0.5 else -1)
        s = self.s + jump
        if s < 0:
            return None, -1.0, True
        if s >= self.n_states:
            return None, 1.0, True
        self.s = s
        return s, 0.0, False


def aggregate(s, n_groups=10, n_states=1000):
    """State aggregation: a one-hot vector saying which block of states s falls in."""
    x = np.zeros(n_groups)
    x[s * n_groups // n_states] = 1.0
    return x


def run_gradient_mc(episodes=4000, alpha=0.0005, seed=0):
    env, w = RandomWalk1000(seed), np.zeros(10)
    for _ in range(episodes):
        s, done, xs, rs = env.reset(), False, [], []
        while not done:
            xs.append(aggregate(s))
            s, r, done = env.step()
            rs.append(r)
        gradient_mc_update(w, xs, rs, alpha)
    return w


def run_td0(episodes=2000, alpha=0.01, seed=0):
    env, w = RandomWalk1000(seed), np.zeros(10)
    for _ in range(episodes):
        s, done = env.reset(), False
        while not done:
            s2, r, done = env.step()
            td0_update(w, aggregate(s), r, None if done else aggregate(s2), done, alpha)
            s = s2
    return w


def true_values():
    """Exact v(s) for the random walk under its (only) policy, by solving v = r + P v."""
    n = RandomWalk1000.n_states
    P, r = np.zeros((n, n)), np.zeros(n)
    for s in range(n):
        for jump in range(1, 101):
            for s2 in (s + jump, s - jump):
                if s2 < 0:
                    r[s] -= 1 / 200
                elif s2 >= n:
                    r[s] += 1 / 200
                else:
                    P[s, s2] += 1 / 200
    return np.linalg.solve(np.eye(n) - P, r)


if __name__ == "__main__":
    v_true = true_values()
    w_mc = run_gradient_mc()
    w_td = run_td0()
    states = range(0, 1000, 10)
    rlhud.plot([v_true[s] for s in states], label="true v(s)", title="Random walk: value vs state (every 10th state)")
    rlhud.plot([v_hat(w_mc, aggregate(s)) for s in states], label="gradient MC", title="Random walk: value vs state (every 10th state)")
    rlhud.plot([v_hat(w_td, aggregate(s)) for s in states], label="semi-gradient TD(0)", title="Random walk: value vs state (every 10th state)")
    rmse = lambda w: float(np.sqrt(np.mean([(v_hat(w, aggregate(s)) - v_true[s]) ** 2 for s in range(1000)])))
    rlhud.stat("RMSE gradient MC", rmse(w_mc))
    rlhud.stat("RMSE TD(0)", rmse(w_td))
    rlhud.bars(w_mc, labels=[f"{100*i}-{100*i+99}" for i in range(10)], title="Gradient MC weights (one per group)")

In [ ]:
#@title ✅ **Check** · Linear values: gradient MC & semi-gradient TD(0) (+120 XP) — tests the code you last ran above
rlcolab.check("08-function-approximation/01-linear-value-updates")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

For a linear function the gradient ∇v̂(s, w) is simply the feature vector x(s), so every update has the shape `w += alpha * error * x`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

gradient_mc_update: first build the list of returns backwards (G = r + gamma * G), then loop over zip(xs, returns) and apply the update.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

td0_update: `target = r if terminated else r + gamma * v_hat(w, x_next)`; `delta = target - v_hat(w, x)`; `w += alpha * delta * x`; `return delta`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("08-function-approximation/01-linear-value-updates")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 08-function-approximation/01-linear-value-updates
# Reference solution
import numpy as np
import rlhud


# ---------------------------------------------------------------- your code
def v_hat(w, x):
    """Linear value estimate: v̂(s, w) = wᵀ x(s)."""
    return float(np.dot(w, x))


def gradient_mc_update(w, xs, rewards, alpha, gamma=1.0):
    """Gradient Monte Carlo for ONE finished episode. Updates w in place and returns it.

    xs      : feature vectors [x(S0), x(S1), ..., x(S_{T-1})]
    rewards : [R1, R2, ..., RT]   (rewards[t] follows the action taken in S_t)
    For every t:  w ← w + α [Gₜ − v̂(Sₜ, w)] ∇v̂(Sₜ, w),  and for a linear v̂, ∇v̂ = x(Sₜ).
    """
    G = 0.0
    returns = np.zeros(len(rewards))
    for t in reversed(range(len(rewards))):      # returns-to-go, backwards
        G = rewards[t] + gamma * G
        returns[t] = G
    for x, G in zip(xs, returns):                # then one SGD step per visited state
        w += alpha * (G - v_hat(w, x)) * x
    return w


def td0_update(w, x, r, x_next, terminated, alpha, gamma=1.0):
    """Semi-gradient TD(0) for one transition S --r--> S'. Updates w in place, returns δ.

    target = r                      if the episode terminated at S'
             r + γ v̂(S', w)         otherwise          (no gradient flows through the target)
    """
    target = r if terminated else r + gamma * v_hat(w, x_next)
    delta = target - v_hat(w, x)
    w += alpha * delta * x
    return delta


# ---------------------------------------------------------------- provided: the 1000-state random walk
class RandomWalk1000:
    """States 0..999, start in the middle. Each step jumps 1..100 states left or right
    (uniformly). Falling off the left edge ends the episode with reward −1, the right edge +1.
    All other rewards are 0. Sutton & Barto, Example 9.1."""

    n_states = 1000

    def __init__(self, seed=None):
        self.rng = np.random.default_rng(seed)

    def reset(self):
        self.s = 499
        return self.s

    def step(self):
        jump = int(self.rng.integers(1, 101)) * (1 if self.rng.random() < 0.5 else -1)
        s = self.s + jump
        if s < 0:
            return None, -1.0, True
        if s >= self.n_states:
            return None, 1.0, True
        self.s = s
        return s, 0.0, False


def aggregate(s, n_groups=10, n_states=1000):
    """State aggregation: a one-hot vector saying which block of states s falls in."""
    x = np.zeros(n_groups)
    x[s * n_groups // n_states] = 1.0
    return x


def run_gradient_mc(episodes=4000, alpha=0.0005, seed=0):
    env, w = RandomWalk1000(seed), np.zeros(10)
    for _ in range(episodes):
        s, done, xs, rs = env.reset(), False, [], []
        while not done:
            xs.append(aggregate(s))
            s, r, done = env.step()
            rs.append(r)
        gradient_mc_update(w, xs, rs, alpha)
    return w


def run_td0(episodes=2000, alpha=0.01, seed=0):
    env, w = RandomWalk1000(seed), np.zeros(10)
    for _ in range(episodes):
        s, done = env.reset(), False
        while not done:
            s2, r, done = env.step()
            td0_update(w, aggregate(s), r, None if done else aggregate(s2), done, alpha)
            s = s2
    return w


def true_values():
    """Exact v(s) for the random walk under its (only) policy, by solving v = r + P v."""
    n = RandomWalk1000.n_states
    P, r = np.zeros((n, n)), np.zeros(n)
    for s in range(n):
        for jump in range(1, 101):
            for s2 in (s + jump, s - jump):
                if s2 < 0:
                    r[s] -= 1 / 200
                elif s2 >= n:
                    r[s] += 1 / 200
                else:
                    P[s, s2] += 1 / 200
    return np.linalg.solve(np.eye(n) - P, r)


if __name__ == "__main__":
    v_true = true_values()
    w_mc = run_gradient_mc()
    w_td = run_td0()
    states = range(0, 1000, 10)
    rlhud.plot([v_true[s] for s in states], label="true v(s)", title="Random walk: value vs state (every 10th state)")
    rlhud.plot([v_hat(w_mc, aggregate(s)) for s in states], label="gradient MC", title="Random walk: value vs state (every 10th state)")
    rlhud.plot([v_hat(w_td, aggregate(s)) for s in states], label="semi-gradient TD(0)", title="Random walk: value vs state (every 10th state)")
    rmse = lambda w: float(np.sqrt(np.mean([(v_hat(w, aggregate(s)) - v_true[s]) ** 2 for s in range(1000)])))
    rlhud.stat("RMSE gradient MC", rmse(w_mc))
    rlhud.stat("RMSE TD(0)", rmse(w_td))
    rlhud.bars(w_mc, labels=[f"{100*i}-{100*i+99}" for i in range(10)], title="Gradient MC weights (one per group)")

## 03 · Features: where the knowledge goes

Some classic choices for a continuous state s = (s₁, …, s_k):

- **State aggregation** — chop the space into cells, one-hot per cell. Simple; values jump at cell boundaries.
- **Polynomials** — 1, s₁, s₂, s₁s₂, s₁², … Global: every feature affects every state. Unstable in high degree.
- **Radial basis functions** — xᵢ(s) = exp(−‖s − cᵢ‖² / 2σ²). Smooth, local, but you must place the centres.
- **Tile coding** — several overlapping, *shifted* grids. Binary, sparse, local and blazingly fast.
- **Learned features** — a neural network learns x(s) itself. That is the next lesson.

> **💡 TIP** · **Normalise your inputs**
>
> CartPole's velocity is ±2, MountainCar's is ±0.07. Unscaled features with very different ranges
> make a single step size α far too big along one direction and far too small along another.
> Rescale every input to roughly [−1, 1] or [0, 1] before building features.

## 04 · Tile coding

Lay a grid ("tiling") over the state space: a state activates exactly **one** tile. Now lay
several more grids, each shifted by a fraction of a tile. A state activates one tile **per
tiling** — with 8 tilings, 8 of the (say) 648 features are 1 and the rest are 0.

```
                       x
tiling 0  |....|....|....|....|       x is in cell 2
tiling 1    |....|....|....|....|     (shifted ⅖ tile)  cell 2
tiling 2      |....|....|....|....|   (shifted ⅘ tile)  cell 1
```

Two states **share** a tile in every tiling where they fall in the same cell. Nearby states
share most of their tiles (generalisation); distant ones share none (discrimination). The
resolution is finer than any single grid — 8 tilings of width w distinguish positions to w/8 —
while generalisation spreads about one tile width.

Two practical rules:

1. **Step size per tiling.** Exactly n features are active, so wᵀx is a sum of n weights. Use
   α = α₀ / n so that one update moves the estimate by about α₀ of the error, whatever n is.
2. **Asymmetric offsets.** Shifting tiling *i* by i/n along *every* dimension lines all tilings
   up on the diagonal. Shifting by (1, 3, 5, …) · i/n per dimension spreads them out.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("08-function-approximation#1", {"q": "A tile coder has 8 tilings. You want each update to correct about half of the TD error. Which per-weight step size?", "options": ["α = 0.5", "α = 0.5 / 8", "α = 0.5 · 8", "α = 1 / 8²"], "answer": 1, "explain": "Eight weights are active and each moves by α·δ, so the estimate moves by 8·α·δ. Setting α = 0.5/8 makes it move by 0.5·δ."})

---
### 🧪 Exercise 2 · Build a tile coder · ⭐ **120 XP**

Implement `TileCoder.active(x)`: return the index of the one active tile in **each** of the `n_tilings` tilings for a continuous point `x`.

The constructor has already prepared everything you need:
- `self.low`, `self.high` — the box the state lives in
- `self.tiles` — tiles per dimension; each tiling has `tiles + 1` per dimension so a shifted grid still covers the box (`self.shape`)
- `self.offsets[i]` — how far tiling *i* is shifted, in units of one tile
- `self.tiles_per_tiling` — size of each tiling's block of feature indices

Steps: rescale so one tile has width 1, add each tiling's offset, `floor`, clip into the grid, flatten row-major, and add `i · tiles_per_tiling`.

Run the demo to see how much each state *shares* with one point, and a sine wave fitted by plain linear SGD on tile features.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 08-function-approximation/02-tile-coder
import numpy as np
import rlhud


class TileCoder:
    """Tile coding for a box-shaped continuous space.

    tc = TileCoder(low=[-1.2, -0.07], high=[0.6, 0.07], n_tilings=8, tiles_per_dim=8)
    idx = tc.active([-0.5, 0.01])      # -> array of n_tilings feature indices (one per tiling)
    x   = tc.features([-0.5, 0.01])    # -> binary vector of length tc.n_features

    Each tiling is a grid of `tiles_per_dim` tiles per dimension, shifted by a different
    fraction of a tile. Each grid gets one extra row/column of tiles so the shifted grid
    still covers the whole box.
    """

    def __init__(self, low, high, n_tilings=8, tiles_per_dim=8):
        self.low = np.asarray(low, dtype=float)
        self.high = np.asarray(high, dtype=float)
        self.n_tilings = n_tilings
        self.tiles = tiles_per_dim
        self.dims = len(self.low)
        self.shape = (tiles_per_dim + 1,) * self.dims          # tiles per tiling, per dimension
        self.tiles_per_tiling = (tiles_per_dim + 1) ** self.dims
        self.n_features = n_tilings * self.tiles_per_tiling
        # offsets[i, k] = how far tiling i is shifted along dimension k, in units of one tile.
        # Asymmetric shifts (1, 3, 5, ... × i / n) avoid all tilings lining up on the diagonal.
        k = np.arange(self.dims)
        self.offsets = (np.arange(n_tilings)[:, None] * (2 * k + 1)[None, :] / n_tilings) % 1.0

    def active(self, x):
        """Indices of the n_tilings active tiles for point x (one per tiling)."""
        # TODO
        # 1) rescale x so that one tile has width 1:  (x − low) / (high − low) · tiles_per_dim
        # 2) for each tiling i: coords = floor(scaled + self.offsets[i]), clipped to 0 … self.tiles
        # 3) flatten coords in row-major order (np.ravel_multi_index(coords, self.shape))
        #    and add i · self.tiles_per_tiling so every tiling has its own block of indices
        raise NotImplementedError

    def features(self, x):
        """The same information as a binary feature vector x(s)."""
        v = np.zeros(self.n_features)
        v[self.active(x)] = 1.0
        return v


if __name__ == "__main__":
    # How much do two states "share"? Overlap of active tiles between the centre of the
    # MountainCar box and every other point = how much an update at one moves the other.
    tc = TileCoder(low=[-1.2, -0.07], high=[0.6, 0.07], n_tilings=8, tiles_per_dim=8)
    centre = set(tc.active([-0.3, 0.0]).tolist())
    pos = np.linspace(-1.2, 0.6, 25)
    vel = np.linspace(0.07, -0.07, 15)            # top row = fast to the right
    overlap = [[len(centre & set(tc.active([p, v]).tolist())) for p in pos] for v in vel]
    rlhud.grid(overlap, title="Shared tiles with the point (−0.3, 0.0)", fmt=".0f", cmap="heat")

    # Generalisation in 1-D: fit a sine with a single linear SGD pass over tile features.
    tc1 = TileCoder(low=[0.0], high=[1.0], n_tilings=8, tiles_per_dim=10)
    rng = np.random.default_rng(0)
    w = np.zeros(tc1.n_features)
    alpha = 0.1 / tc1.n_tilings                   # step size per active tile
    for _ in range(3000):
        s = rng.random()
        idx = tc1.active([s])
        w[idx] += alpha * (np.sin(2 * np.pi * s) - w[idx].sum())
    grid_x = np.linspace(0, 1, 101)
    rlhud.plot([np.sin(2 * np.pi * s) for s in grid_x], label="target sin(2πs)", title="Tile-coded fit")
    rlhud.plot([w[tc1.active([s])].sum() for s in grid_x], label="tile coding (8×10)", title="Tile-coded fit")
    rlhud.stat("Features", tc.n_features)

In [ ]:
#@title ✅ **Check** · Build a tile coder (+120 XP) — tests the code you last ran above
rlcolab.check("08-function-approximation/02-tile-coder")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`scaled = (x - self.low) / (self.high - self.low) * self.tiles` — now 1 unit = 1 tile.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

You can do all tilings at once: `coords = np.floor(scaled[None, :] + self.offsets).astype(int)` has shape (n_tilings, dims). Clip it with `np.clip(coords, 0, self.tiles)`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

`flat = np.ravel_multi_index(coords.T, self.shape)` then `return np.arange(self.n_tilings) * self.tiles_per_tiling + flat`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("08-function-approximation/02-tile-coder")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 08-function-approximation/02-tile-coder
# Reference solution
import numpy as np
import rlhud


class TileCoder:
    """Tile coding for a box-shaped continuous space.

    tc = TileCoder(low=[-1.2, -0.07], high=[0.6, 0.07], n_tilings=8, tiles_per_dim=8)
    idx = tc.active([-0.5, 0.01])      # -> array of n_tilings feature indices (one per tiling)
    x   = tc.features([-0.5, 0.01])    # -> binary vector of length tc.n_features

    Each tiling is a grid of `tiles_per_dim` tiles per dimension, shifted by a different
    fraction of a tile. Each grid gets one extra row/column of tiles so the shifted grid
    still covers the whole box.
    """

    def __init__(self, low, high, n_tilings=8, tiles_per_dim=8):
        self.low = np.asarray(low, dtype=float)
        self.high = np.asarray(high, dtype=float)
        self.n_tilings = n_tilings
        self.tiles = tiles_per_dim
        self.dims = len(self.low)
        self.shape = (tiles_per_dim + 1,) * self.dims          # tiles per tiling, per dimension
        self.tiles_per_tiling = (tiles_per_dim + 1) ** self.dims
        self.n_features = n_tilings * self.tiles_per_tiling
        # offsets[i, k] = how far tiling i is shifted along dimension k, in units of one tile.
        # Asymmetric shifts (1, 3, 5, ... × i / n) avoid all tilings lining up on the diagonal.
        k = np.arange(self.dims)
        self.offsets = (np.arange(n_tilings)[:, None] * (2 * k + 1)[None, :] / n_tilings) % 1.0

    def active(self, x):
        """Indices of the n_tilings active tiles for point x (one per tiling)."""
        # 1) rescale x so that one tile has width 1:  0 … tiles_per_dim across the box
        scaled = (np.asarray(x, dtype=float) - self.low) / (self.high - self.low) * self.tiles
        # 2) shift by each tiling's offset and take the integer tile coordinate
        coords = np.floor(scaled[None, :] + self.offsets).astype(int)     # (n_tilings, dims)
        coords = np.clip(coords, 0, self.tiles)                            # stay inside the grid
        # 3) flatten (row-major / C order) and give every tiling its own block of indices
        flat = np.ravel_multi_index(coords.T, self.shape)
        return np.arange(self.n_tilings) * self.tiles_per_tiling + flat

    def features(self, x):
        """The same information as a binary feature vector x(s)."""
        v = np.zeros(self.n_features)
        v[self.active(x)] = 1.0
        return v


if __name__ == "__main__":
    # How much do two states "share"? Overlap of active tiles between the centre of the
    # MountainCar box and every other point = how much an update at one moves the other.
    tc = TileCoder(low=[-1.2, -0.07], high=[0.6, 0.07], n_tilings=8, tiles_per_dim=8)
    centre = set(tc.active([-0.3, 0.0]).tolist())
    pos = np.linspace(-1.2, 0.6, 25)
    vel = np.linspace(0.07, -0.07, 15)            # top row = fast to the right
    overlap = [[len(centre & set(tc.active([p, v]).tolist())) for p in pos] for v in vel]
    rlhud.grid(overlap, title="Shared tiles with the point (−0.3, 0.0)", fmt=".0f", cmap="heat")

    # Generalisation in 1-D: fit a sine with a single linear SGD pass over tile features.
    tc1 = TileCoder(low=[0.0], high=[1.0], n_tilings=8, tiles_per_dim=10)
    rng = np.random.default_rng(0)
    w = np.zeros(tc1.n_features)
    alpha = 0.1 / tc1.n_tilings                   # step size per active tile
    for _ in range(3000):
        s = rng.random()
        idx = tc1.active([s])
        w[idx] += alpha * (np.sin(2 * np.pi * s) - w[idx].sum())
    grid_x = np.linspace(0, 1, 101)
    rlhud.plot([np.sin(2 * np.pi * s) for s in grid_x], label="target sin(2πs)", title="Tile-coded fit")
    rlhud.plot([w[tc1.active([s])].sum() for s in grid_x], label="tile coding (8×10)", title="Tile-coded fit")
    rlhud.stat("Features", tc.n_features)

## 05 · Control: semi-gradient SARSA

For control we approximate **action values** q̂(s, a, w) and act ε-greedily, exactly as in the
tabular case. The SARSA update becomes

```text
w ← w + α [ Rₜ₊₁ + γ q̂(Sₜ₊₁, Aₜ₊₁, w) − q̂(Sₜ, Aₜ, w) ] ∇q̂(Sₜ, Aₜ, w)
```

With few discrete actions, the easiest design is **one weight vector per action**: q̂(s, a) =
wₐᵀ x(s). Only the row of the action you took gets updated.

### Mountain Car

An under-powered car sits in a valley. Full throttle is not enough to climb the right hill; it
has to back up the left slope first to build momentum. Reward is −1 per step until the flag,
so the return is minus the number of steps. The state is (position, velocity) — continuous.

Two details make this work with ε = 0:

- **Optimistic initialisation.** All weights start at 0, but every real return is negative.
  Untried actions look better than tried ones, so the greedy agent explores systematically.
- **Truncation is not termination.** Our episodes are cut off after `max_steps`. That cut is
  about *our* patience, not the car: the state still has a future. On truncation, keep
  bootstrapping from q̂(S′, A′). Only a real `terminated` gets target = r.

> **⚠️ WARNING** · **The truncation bug**
>
> Treating a time-limit cut as a terminal state teaches the agent that states near the time
> limit are worth 0 — a lie it will happily believe. In Gymnasium-style APIs, bootstrap
> unless `terminated` is True, regardless of `truncated`.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("08-function-approximation#2", {"q": "In MountainCar, why does a purely greedy (ε = 0) agent with zero-initialised weights still explore?", "options": ["MountainCar is deterministic, so exploration is not needed", "Every true value is negative, so actions it has not tried yet look better (optimism)", "Tile coding adds random noise", "SARSA always explores"], "answer": 1, "explain": "Each visit lowers the estimate of the action taken (rewards are −1). Untouched actions keep their optimistic 0 and get picked next. This drives systematic exploration without any randomness."})

---
### 🧪 Exercise 3 · Semi-gradient SARSA on Mountain Car · ⭐ **200 XP**

Teach an under-powered car to rock back and forth until it can climb the hill — using **linear function approximation over tile-coded features**.

Implement:
- `q_values(w, idx)` → q̂(s, a) for all actions. `w` has shape `(n_actions, n_features)`; with binary tile features, wₐᵀx(s) is just the sum of the active weights.
- `epsilon_greedy(q, epsilon, rng)` with **random tie-breaking** (all weights start at 0, so ties are everywhere at first).
- `sarsa_update(w, idx, a, r, idx_next, a_next, terminated, alpha, gamma)` → target r + γ q̂(S′, A′) (or just r if terminated), δ = target − q̂(S, A), add αδ to the active weights of action `a`. Return δ.

The training loop is provided: read it — note how a *truncated* episode still bootstraps. The check trains for 150 episodes (seeded) and expects the car to reach the flag in under 200 steps on average. The demo trains longer and draws the cost-to-go surface.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 08-function-approximation/03-mountain-car-sarsa
import numpy as np
import rlhud
from rlenvs import MountainCar


# ---------------------------------------------------------------- provided: tile coder (previous exercise)
class TileCoder:
    def __init__(self, low, high, n_tilings=8, tiles_per_dim=8):
        self.low, self.high = np.asarray(low, float), np.asarray(high, float)
        self.n_tilings, self.tiles, self.dims = n_tilings, tiles_per_dim, len(self.low)
        self.shape = (tiles_per_dim + 1,) * self.dims
        self.tiles_per_tiling = (tiles_per_dim + 1) ** self.dims
        self.n_features = n_tilings * self.tiles_per_tiling
        k = np.arange(self.dims)
        self.offsets = (np.arange(n_tilings)[:, None] * (2 * k + 1)[None, :] / n_tilings) % 1.0

    def active(self, x):
        scaled = (np.asarray(x, float) - self.low) / (self.high - self.low) * self.tiles
        coords = np.clip(np.floor(scaled[None, :] + self.offsets).astype(int), 0, self.tiles)
        return np.arange(self.n_tilings) * self.tiles_per_tiling + np.ravel_multi_index(coords.T, self.shape)


# ---------------------------------------------------------------- your code
def q_values(w, idx):
    """q̂(s, a, w) for every action at once.

    w   : weights, shape (n_actions, n_features) — one linear function per action
    idx : active tile indices of s (the features are binary, so wᵀx(s) = sum of active weights)
    returns an array of shape (n_actions,)
    """
    # TODO: one line of numpy indexing
    raise NotImplementedError


def epsilon_greedy(q, epsilon, rng):
    """Random action with probability epsilon, otherwise a greedy one (ties broken at random)."""
    # TODO: use rng.random(), rng.integers(...) and rng.choice(np.flatnonzero(q == q.max()))
    raise NotImplementedError


def sarsa_update(w, idx, a, r, idx_next, a_next, terminated, alpha, gamma=1.0):
    """One semi-gradient SARSA step. Updates w in place and returns the TD error δ.

    target = r                                  if terminated
             r + γ q̂(S′, A′, w)                 otherwise
    w[a] ← w[a] + α δ ∇q̂  — the gradient is 1 on the active tiles of S and 0 elsewhere.
    """
    # TODO: target, δ, then add α·δ to the active weights of action a only
    raise NotImplementedError


# ---------------------------------------------------------------- provided: the training loop
def train(episodes=150, seed=0, alpha=0.5, epsilon=0.0, gamma=1.0, n_tilings=8, tiles=8,
          max_steps=1000, log=None):
    """Semi-gradient SARSA on MountainCar. Returns (w, tile_coder, steps_per_episode).

    alpha is the *total* step size: each of the n_tilings active weights moves by alpha/n_tilings.
    epsilon=0 works here because q̂ starts at 0 while every real return is negative —
    "optimistic initialisation" makes untried actions look attractive.
    """
    rng = np.random.default_rng(seed)
    env = MountainCar(max_steps=max_steps, seed=seed)
    tc = TileCoder(env.low, env.high, n_tilings, tiles)
    w = np.zeros((env.n_actions, tc.n_features))
    step = alpha / n_tilings
    steps_per_episode = []
    for ep in range(episodes):
        obs, _ = env.reset()
        idx = tc.active(obs)
        a = epsilon_greedy(q_values(w, idx), epsilon, rng)
        n = 0
        while True:
            obs, r, terminated, truncated, _ = env.step(a)
            n += 1
            if terminated:
                sarsa_update(w, idx, a, r, None, None, True, step, gamma)
                break
            idx2 = tc.active(obs)
            a2 = epsilon_greedy(q_values(w, idx2), epsilon, rng)
            # A time-limit cut is NOT a real ending: still bootstrap from q̂(S′, A′).
            sarsa_update(w, idx, a, r, idx2, a2, False, step, gamma)
            idx, a = idx2, a2
            if truncated:
                break
        steps_per_episode.append(n)
        if log:
            log(ep, n)
    return w, tc, steps_per_episode


def greedy_steps(w, tc, seed=123, max_steps=1000):
    """Run one greedy episode and return how many steps it took to reach the flag."""
    env = MountainCar(max_steps=max_steps)
    obs, _ = env.reset(seed=seed)
    for t in range(1, max_steps + 1):
        obs, _, terminated, truncated, _ = env.step(int(np.argmax(q_values(w, tc.active(obs)))))
        if terminated or truncated:
            return t
    return max_steps


if __name__ == "__main__":
    live = rlhud.chart("Steps to reach the flag (lower is better)", smooth=10, xlabel="episode")
    w, tc, steps = train(episodes=300, seed=0, log=lambda ep, n: (live.add("SARSA", n), rlhud.progress(ep, 300)))
    live.flush()
    rlhud.stat("Mean steps, last 20 episodes", float(np.mean(steps[-20:])))
    rlhud.stat("Greedy run (steps)", greedy_steps(w, tc))

    # The famous "cost-to-go" picture: −max_a q̂(s, a) over the whole state space.
    pos = np.linspace(-1.2, 0.6, 24)
    vel = np.linspace(0.07, -0.07, 16)
    ctg = [[-q_values(w, tc.active([p, v])).max() for p in pos] for v in vel]
    rlhud.grid(ctg, title="Cost-to-go −max q̂  (x: position →, y: velocity, top = +0.07)", fmt=".0f", cmap="heat")

In [ ]:
#@title ✅ **Check** · Semi-gradient SARSA on Mountain Car (+200 XP) — tests the code you last ran above
rlcolab.check("08-function-approximation/03-mountain-car-sarsa")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

q_values: fancy indexing `w[:, idx]` has shape (n_actions, n_tilings); sum over the last axis.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

sarsa_update: `target = r if terminated else r + gamma * w[a_next, idx_next].sum()` and `delta = target - w[a, idx].sum()`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

Then `w[a, idx] += alpha * delta` and `return delta`. Because each tile index appears once per tiling, the in-place add is safe.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("08-function-approximation/03-mountain-car-sarsa")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 08-function-approximation/03-mountain-car-sarsa
# Reference solution
import numpy as np
import rlhud
from rlenvs import MountainCar


# ---------------------------------------------------------------- provided: tile coder (previous exercise)
class TileCoder:
    def __init__(self, low, high, n_tilings=8, tiles_per_dim=8):
        self.low, self.high = np.asarray(low, float), np.asarray(high, float)
        self.n_tilings, self.tiles, self.dims = n_tilings, tiles_per_dim, len(self.low)
        self.shape = (tiles_per_dim + 1,) * self.dims
        self.tiles_per_tiling = (tiles_per_dim + 1) ** self.dims
        self.n_features = n_tilings * self.tiles_per_tiling
        k = np.arange(self.dims)
        self.offsets = (np.arange(n_tilings)[:, None] * (2 * k + 1)[None, :] / n_tilings) % 1.0

    def active(self, x):
        scaled = (np.asarray(x, float) - self.low) / (self.high - self.low) * self.tiles
        coords = np.clip(np.floor(scaled[None, :] + self.offsets).astype(int), 0, self.tiles)
        return np.arange(self.n_tilings) * self.tiles_per_tiling + np.ravel_multi_index(coords.T, self.shape)


# ---------------------------------------------------------------- your code
def q_values(w, idx):
    """q̂(s, a, w) for every action at once.

    w   : weights, shape (n_actions, n_features) — one linear function per action
    idx : active tile indices of s (the features are binary, so wᵀx(s) = sum of active weights)
    returns an array of shape (n_actions,)
    """
    return w[:, idx].sum(axis=1)


def epsilon_greedy(q, epsilon, rng):
    """Random action with probability epsilon, otherwise a greedy one (ties broken at random)."""
    if rng.random() < epsilon:
        return int(rng.integers(len(q)))
    return int(rng.choice(np.flatnonzero(q == q.max())))


def sarsa_update(w, idx, a, r, idx_next, a_next, terminated, alpha, gamma=1.0):
    """One semi-gradient SARSA step. Updates w in place and returns the TD error δ.

    target = r                                  if terminated
             r + γ q̂(S′, A′, w)                 otherwise
    w[a] ← w[a] + α δ ∇q̂  — the gradient is 1 on the active tiles of S and 0 elsewhere.
    """
    target = r if terminated else r + gamma * w[a_next, idx_next].sum()
    delta = target - w[a, idx].sum()
    w[a, idx] += alpha * delta
    return delta


# ---------------------------------------------------------------- provided: the training loop
def train(episodes=150, seed=0, alpha=0.5, epsilon=0.0, gamma=1.0, n_tilings=8, tiles=8,
          max_steps=1000, log=None):
    """Semi-gradient SARSA on MountainCar. Returns (w, tile_coder, steps_per_episode).

    alpha is the *total* step size: each of the n_tilings active weights moves by alpha/n_tilings.
    epsilon=0 works here because q̂ starts at 0 while every real return is negative —
    "optimistic initialisation" makes untried actions look attractive.
    """
    rng = np.random.default_rng(seed)
    env = MountainCar(max_steps=max_steps, seed=seed)
    tc = TileCoder(env.low, env.high, n_tilings, tiles)
    w = np.zeros((env.n_actions, tc.n_features))
    step = alpha / n_tilings
    steps_per_episode = []
    for ep in range(episodes):
        obs, _ = env.reset()
        idx = tc.active(obs)
        a = epsilon_greedy(q_values(w, idx), epsilon, rng)
        n = 0
        while True:
            obs, r, terminated, truncated, _ = env.step(a)
            n += 1
            if terminated:
                sarsa_update(w, idx, a, r, None, None, True, step, gamma)
                break
            idx2 = tc.active(obs)
            a2 = epsilon_greedy(q_values(w, idx2), epsilon, rng)
            # A time-limit cut is NOT a real ending: still bootstrap from q̂(S′, A′).
            sarsa_update(w, idx, a, r, idx2, a2, False, step, gamma)
            idx, a = idx2, a2
            if truncated:
                break
        steps_per_episode.append(n)
        if log:
            log(ep, n)
    return w, tc, steps_per_episode


def greedy_steps(w, tc, seed=123, max_steps=1000):
    """Run one greedy episode and return how many steps it took to reach the flag."""
    env = MountainCar(max_steps=max_steps)
    obs, _ = env.reset(seed=seed)
    for t in range(1, max_steps + 1):
        obs, _, terminated, truncated, _ = env.step(int(np.argmax(q_values(w, tc.active(obs)))))
        if terminated or truncated:
            return t
    return max_steps


if __name__ == "__main__":
    live = rlhud.chart("Steps to reach the flag (lower is better)", smooth=10, xlabel="episode")
    w, tc, steps = train(episodes=300, seed=0, log=lambda ep, n: (live.add("SARSA", n), rlhud.progress(ep, 300)))
    live.flush()
    rlhud.stat("Mean steps, last 20 episodes", float(np.mean(steps[-20:])))
    rlhud.stat("Greedy run (steps)", greedy_steps(w, tc))

    # The famous "cost-to-go" picture: −max_a q̂(s, a) over the whole state space.
    pos = np.linspace(-1.2, 0.6, 24)
    vel = np.linspace(0.07, -0.07, 16)
    ctg = [[-q_values(w, tc.active([p, v])).max() for p in pos] for v in vel]
    rlhud.grid(ctg, title="Cost-to-go −max q̂  (x: position →, y: velocity, top = +0.07)", fmt=".0f", cmap="heat")

After training, the demo draws −max q̂ over the state space: a spiral-shaped "cost-to-go"
landscape. The low-cost ridge follows the swinging trajectories the car learned.

## 06 · The deadly triad

Function approximation does not always converge. Sutton & Barto identify three ingredients
that together can make values **diverge to infinity**, even on tiny problems with linear features:

1. **Function approximation** — updates generalise to other states.
2. **Bootstrapping** — targets built from the current estimates (TD, DP).
3. **Off-policy learning** — training on a distribution of states different from the policy being evaluated (e.g. Q-learning's greedy target with ε-greedy data).

Any two are fine. All three together are dangerous. Baird's counterexample (7 states, linear
features, all rewards zero!) sends the weights to infinity under semi-gradient off-policy TD.

> **⚠️ WARNING** · **Deep Q-learning lives inside the deadly triad**
>
> DQN (next lesson) uses a neural network (1), bootstraps (2) and learns off-policy from a replay
> buffer (3). Its tricks — target networks, replay, Huber loss — exist precisely to tame this
> instability. When deep RL blows up, the triad is the first suspect.

> **🔑 KEY IDEA** · **Generalisation is the point — and the risk**
>
> A function approximator lets an update in one state inform similar states. With linear
> features, on-policy semi-gradient TD is stable and converges near the best approximation.
> Add off-policy bootstrapping and that guarantee disappears.

## 07 · Debugging function approximation

- **Test on a problem with known values first** (like the random walk): compare v̂ with the truth before trusting a control result.
- **Watch the weight norm.** ‖w‖ growing without bound means divergence: lower α, check the off-policy setup.
- **Scale α by the number of active features** (tile coding) or by feature magnitude (‖x‖²).
- **Bootstrap on truncation**, never on termination.
- **Seed everything** and run more than one seed — a single lucky run proves nothing.
- **Plot something interpretable**: the value surface, the steps per episode, a greedy rollout.

In [ ]:
#@title 🧠 **Knowledge check 4** · +25 XP — run, then pick an answer
rlcolab.quiz("08-function-approximation#3", {"q": "Which combination makes up the deadly triad?", "options": ["Exploration, exploitation and discounting", "Function approximation, bootstrapping and off-policy training", "Large α, small γ and ε-greedy", "Tile coding, Monte Carlo and SARSA"], "answer": 1, "explain": "Any two are safe; all three together can diverge. Monte Carlo (no bootstrapping) or on-policy linear TD avoid it."})

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("08-function-approximation")